# 06 – Model Training: Quantiles (τ = 0.1 and τ = 0.9)

**Purpose:** Train two PyTorch neural networks using pinball (quantile) loss to predict the 10th and 90th percentiles of yield.

**Steps:**
1. Load pre‑processed train/val arrays from feature engineering.
2. Apply log1p transform to target.
3. Define the same neural network architecture used earlier.
4. Train with pinball loss at τ = 0.1 (lower) and τ = 0.9 (upper).
5. Save models to `backend/models/`.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os

os.makedirs('../backend/models', exist_ok=True)

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [2]:
X_train = np.load('../data/X_train.npy')
y_train = np.load('../data/y_train.npy')
X_val = np.load('../data/X_val.npy')
y_val = np.load('../data/y_val.npy')

y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)

X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train_log, dtype=torch.float32).view(-1, 1).to(device)
X_val_t = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val_t = torch.tensor(y_val_log, dtype=torch.float32).view(-1, 1).to(device)

print(f"Train: {X_train_t.shape}, Val: {X_val_t.shape}")

Train: torch.Size([13667, 13]), Val: torch.Size([2929, 13])


In [3]:
class YieldPredictor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)

input_dim = X_train.shape[1]
print(f"Input dimension: {input_dim}")

Input dimension: 13


In [4]:
def quantile_loss(y_pred, y_true, tau):
    error = y_true - y_pred
    loss = torch.max(tau * error, (tau - 1) * error)
    return loss.mean()

In [5]:
def train_quantile_model(tau, X_train, y_train, X_val, y_val, epochs=150, lr=0.001, patience=20):
    model = YieldPredictor(input_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = quantile_loss(outputs, y_train, tau)
        loss.backward()
        optimizer.step()
        
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val)
            val_loss = quantile_loss(val_outputs, y_val, tau)
        
        scheduler.step(val_loss)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
        
        if (epoch+1) % 20 == 0:
            print(f"τ={tau} Epoch {epoch+1}, Loss: {loss.item():.4f}, Val Loss: {val_loss.item():.4f}")
    
    model.load_state_dict(best_state)
    return model, best_val_loss

print("Training lower quantile (τ=0.1)...")
model_lower, _ = train_quantile_model(0.1, X_train_t, y_train_t, X_val_t, y_val_t)
torch.save(model_lower.state_dict(), '../backend/models/yield_model_lower.pt')
print("Saved lower model.")

print("\nTraining upper quantile (τ=0.9)...")
model_upper, _ = train_quantile_model(0.9, X_train_t, y_train_t, X_val_t, y_val_t)
torch.save(model_upper.state_dict(), '../backend/models/yield_model_upper.pt')
print("Saved upper model.")

Training lower quantile (τ=0.1)...
τ=0.1 Epoch 20, Loss: 0.0779, Val Loss: 0.0824
τ=0.1 Epoch 40, Loss: 0.0692, Val Loss: 0.0694
τ=0.1 Epoch 60, Loss: 0.0645, Val Loss: 0.0619
τ=0.1 Epoch 80, Loss: 0.0612, Val Loss: 0.0574
τ=0.1 Epoch 100, Loss: 0.0594, Val Loss: 0.0556
τ=0.1 Epoch 120, Loss: 0.0577, Val Loss: 0.0536
τ=0.1 Epoch 140, Loss: 0.0567, Val Loss: 0.0527
Saved lower model.

Training upper quantile (τ=0.9)...
τ=0.9 Epoch 20, Loss: 0.2442, Val Loss: 0.3093
τ=0.9 Epoch 40, Loss: 0.1143, Val Loss: 0.0918
τ=0.9 Epoch 60, Loss: 0.0969, Val Loss: 0.0802
τ=0.9 Epoch 80, Loss: 0.0893, Val Loss: 0.0710
τ=0.9 Epoch 100, Loss: 0.0852, Val Loss: 0.0669
τ=0.9 Epoch 120, Loss: 0.0807, Val Loss: 0.0645
τ=0.9 Epoch 140, Loss: 0.0806, Val Loss: 0.0629
Saved upper model.


### Summary

- Lower quantile model → `yield_model_lower.pt`
- Upper quantile model → `yield_model_upper.pt`

**Next:** Run `04_model_evaluation.ipynb` (you already have it) to evaluate all three models (Random Forest median + quantile NNs).